# Acoustic refinement of an area change

A **tapered duct** (a horn / nozzle) is a **composite** element in Nefes: it discretizes a smooth area profile into a chain of compact area changes, each followed by a short duct.
Its **mean flow is exact at any resolution**: An area change is a state-function relation, so the chain reproduces the continuous profile with no truncation error, and refining the segment count $N$ changes nothing in terms of the mean flow.

Nefes internally represents a tapered duct element as a series of ducts and isentropic are change elements, as shown below:

![Network topology](acoustic_refinement_topology.png)

Its **acoustics are a different story.**
A single compact area change is a zero-length impedance discontinuity: it reflects strongly and carries no transit phase.
A *distributed* horn spreads that impedance change over a length comparable to the wavelength, so it reflects far less and adds real propagation delay.
Neither is captured at low $N$, so the **scattering matrix genuinely converges with refinement**, and the compact ($N=1$) limit can be badly wrong.

This notebook measures that convergence and contrasts it with the resolution-independent mean flow, and demonstrates the `refine_grid` and `auto_refine` routines Nefes offers for discretized-type composite elements.

We consider a linear area change from $A_{\text{in}}$ to $A_{\text{out}}$ over a length of $L$.
The tapered duct element accepts the variation of area either as tabulated input or as a callable.

In [8]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nefes
from nefes.elements import catalog as cat
from nefes.elements import auto_refine, grid_refine
from nefes.plotting import use_nefes_theme, COLORWAY

use_nefes_theme()

CFG = nefes.perfect_gas(287.0, 1.4)
L, A_IN, A_OUT = 0.4, 4.0e-3, 2.0e-3  # a 2:1 converging horn, 0.4 m long
PT_IN, P_OUT, T0 = 1.08e5, 1.0e5, 300.0


def area(x):
    "Linear area profile A(x) from the inlet to the outlet."
    return A_IN + (A_OUT - A_IN) * (x / L)


def horn(n):
    "The horn as a Network: inlet -> tapered_duct(N segments) -> outlet."
    return nefes.Network(
        CFG,
        nodes=[
            cat.total_pressure_inlet(PT_IN, T0),
            cat.tapered_duct(area, length=L, n_segments=n, name="horn"),
            cat.pressure_outlet(P_OUT, T0),
        ],
        edges=[(0, 1, A_IN), (1, 2, A_OUT)],
    )

## The mean flow is resolution-independent

Solve the horn at a coarse and a fine resolution: the exit Mach is identical to machine precision.
The lumped area changes are exact, so refinement adds interior sample points but no accuracy.

In [19]:
for n in (1, 2, 8, 32):
    sol = horn(n).solve()
    print(f"N = {n:>3}   exit Mach = {sol.field('M')[1]:.8f}")

N =   1   exit Mach = 0.33340970
N =   2   exit Mach = 0.33340970
N =   8   exit Mach = 0.33340970
N =  32   exit Mach = 0.33340971


## The scattering matrix

The 2-port scattering matrix `S` maps the incoming acoustic waves (inlet + outlet) to the outgoing ones:
$$
\begin{bmatrix} \widehat{g}_a \\ \widehat{f}_b \end{bmatrix}
=
\begin{bmatrix} r_{+} & t_{-} \\ t_{+} & r_{-} \end{bmatrix}
\begin{bmatrix} \widehat{f}_a \\ \widehat{g}_b \end{bmatrix},
$$
We read it between the **inlet edge (0)** and the **outlet edge (1)** across a frequency sweep, at several resolutions.
To do so, we use the `perturbation_response`, which populates a database of forced response results in which scattering information can be extracted.
`|S11|` is the inlet reflection coefficient; watch it collapse as the horn is resolved and drop with frequency, as a longer horn (relative to the wavelength) becomes a better impedance match.

In [26]:
def scatter(n, freqs):
    "Scattering matrix S(inlet edge 0 -> outlet edge 1), shape (n_freq, 2, 2)."
    sol = horn(n).solve()
    resp = sol.perturbation_response(np.atleast_1d(np.asarray(freqs, dtype=float)), excite=("acoustic",))
    return np.asarray(resp.scattering_matrix(0, 1)).reshape(-1, 2, 2)


freqs = np.linspace(40.0, 2000.0, 100)
resolutions = [1, 2, 4, 8, 32, 64]
S = {n: scatter(n, freqs) for n in resolutions}

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, subplot_titles=("inlet reflection  |S11|", "transmission phase  arg(S21)  [deg]")
)
for n in resolutions:
    lbl = "N = 1 (compact)" if n == 1 else f"N = {n}"
    fig.add_scatter(x=freqs, y=np.abs(S[n][:, 0, 0]), name=lbl, row=1, col=1)
    fig.add_scatter(x=freqs, y=np.degrees(np.angle(S[n][:, 1, 0])), name=lbl, showlegend=False, row=2, col=1)
fig.update_xaxes(title_text=r"$f\;[\mathrm{Hz}]$", row=2, col=1)
fig.update_yaxes(title_text="|S11|", row=1, col=1)
fig.update_yaxes(title_text="arg(S21) [deg]", row=2, col=1)
fig.update_layout(height=560, title="Horn scattering matrix vs refinement (compact jump - resolved horn)")
fig.show()

The compact model ($N=1$) is frequency-flat in reflection and badly overpredicts it, and its transmission phase misses the horn's transit delay.
As $N$ grows the reflection collapses and the phase settles.

## How fast does it converge?

At a fixed frequency, the distance of `S(N)` from a fine reference falls roughly like $\mathcal{O}(1/N)$.
This is **first order**, far slower than the mean flow's instant convergence.
So an acoustic quantity needs genuine refinement (several segments per wavelength across the horn), not the one doubling a mean-flow quantity takes.

In [5]:
f0 = 1500.0
S_ref = scatter(64, f0)[0]
Ns = np.array([1, 2, 4, 8, 16, 32])
err = np.array([np.linalg.norm(scatter(n, f0)[0] - S_ref) for n in Ns])

fig = go.Figure()
fig.add_scatter(x=Ns, y=err, mode="lines+markers", name="||S(N) - S_ref||", line=dict(color=COLORWAY[4], width=3))
fig.add_scatter(x=Ns, y=err[0] / Ns, mode="lines", name="O(1/N) guide", line=dict(color="#9aa5b1", dash="dot"))
fig.update_xaxes(title_text="segments N", type="log")
fig.update_yaxes(title_text=f"scattering-matrix error at {f0:.0f} Hz", type="log")
fig.update_layout(title="Scattering matrix converges ~ first order in N")
fig.show()

## Converging an acoustic quantity with the refinement tools

`grid_refine` / `auto_refine` (the discretization-convergence helpers) work on **any** probed quantity including an acoustic one.
Point them at the reflection magnitude at the top frequency of interest and they refine until it settles.

A caveat worth stating: `|S11|` has a shallow plateau at very coarse $N$ before it drops, so a
*loose* tolerance can "converge" prematurely on the plateau.
Converge an acoustic quantity of interest at the frequency and tolerance you actually care about.

In [28]:
# mean-flow quantity of interest: exit Mach -> converges in one doubling
mean_ref = grid_refine(lambda n: horn(n).solve(), 1, lambda s: {"M_exit": float(s.field("M")[1])})
print(f"mean flow  (exit Mach):  worst rel-change 1->2 = {mean_ref.worst:.2e}  -> converged at N=2")


# acoustic quantity of interest: |S11| at 1500 Hz -> needs real refinement
def refl(n):
    return {"absS11": float(np.abs(scatter(n, f0)[0][0, 0]))}


ar = auto_refine(refl, 1, lambda d: d, tol=2e-2, max_refine=10)
print(
    f"acoustics  (|S11| @ {f0:.0f} Hz):  converged={ar.converged}  "
    f"at N={ar.n_final} after {ar.n_refine} refinements  ->  |S11| = {ar.final['absS11']:.4f}"
)

mean flow  (exit Mach):  worst rel-change 1->2 = 2.71e-10  -> converged at N=2
acoustics  (|S11| @ 1500 Hz):  converged=True  at N=64 after 6 refinements  ->  |S11| = 0.0409
